# Retrieval as prompting (offline)

**Session 5 · Track A · local Ollama**

A minimal, offline RAG: local embeddings + numpy cosine + a grounded prompt.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import numpy as np
import ollama  # needs the local Ollama app + `ollama pull nomic-embed-text`
from utils import ask


In [ ]:
DOCS = [
  "The library opens at 9am on weekdays.",
  "Return books within 21 days to avoid a fine.",
  "The cafe on the third floor closes at 5pm.",
]

def embed(t):
    return np.array(ollama.embeddings(model="nomic-embed-text", prompt=t)["embedding"])

DOC_VECS = [embed(d) for d in DOCS]

def retrieve(q, k=2):
    qv = embed(q)
    sims = [float(qv @ v / (np.linalg.norm(qv)*np.linalg.norm(v))) for v in DOC_VECS]
    order = np.argsort(sims)[::-1][:k]
    return [DOCS[i] for i in order]

def answer(q):
    ctx = "\n".join(f"[{i+1}] {c}" for i,c in enumerate(retrieve(q)))
    prompt = f"Answer only from the context, cite [n], or say I do not know.\n\n{ctx}\n\nQ: {q}"
    return ask(prompt)

print(answer("When can I get coffee?"))
print(answer("What is the wifi password?"))  # not in docs -> should refuse

### Worked example

Score the retriever on in-scope questions (should answer, grounded) and out-of-scope questions (should refuse).


In [ ]:
# Worked example: score grounding + refusal on a small labelled set
QUESTIONS = [
    {"input": "When does the library open?",   "expected": "in_scope"},
    {"input": "How long can I keep a book?",   "expected": "in_scope"},
    {"input": "When does the cafe close?",     "expected": "in_scope"},
    {"input": "What is the wifi password?",    "expected": "refuse"},
    {"input": "Who is the head librarian?",    "expected": "refuse"},
]

def refused(ans):
    a = ans.lower()
    return "do not know" in a or "don't know" in a or "i do not" in a

ok = 0
for q in QUESTIONS:
    ans = answer(q["input"])
    got = "refuse" if refused(ans) else "in_scope"
    ok += got == q["expected"]
    print(f"  {got:9} (want {q['expected']:9}) | {q['input']}")
print(f"\ngrounding + refusal: {ok}/{len(QUESTIONS)}")


## Your turn - vary the example

1. Add 3 questions whose answer is only *partly* in the docs. Should the bot answer or refuse?
2. Weaken the grounding instruction and watch refusal accuracy drop.
3. Add a distractor doc close in wording to a real one; does retrieval still pick the right chunk?


In [ ]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
